# 🏠 Exercícios de Modelos Não Supervisionados de Machine Learning

**MBA em Data Science e Analytics (USP/ESALQ)**  
**Prof. Dr. Wilson Tarantin Junior**

---

Este notebook implementa uma **análise híbrida** combinando duas técnicas:

- **ACM (Análise de Correspondência Múltipla)** para as variáveis qualitativas
- **PCA (Análise de Componentes Principais)** para as variáveis quantitativas + coordenadas extraídas da ACM

Dataset: preços de casas — [Kaggle: Jiff's House Price Prediction](https://www.kaggle.com/datasets/elakiricoder/jiffs-house-price-prediction-dataset)

## 📦 Instalação dos Pacotes

Execute diretamente no **terminal/console** (não dentro do notebook):

```bash
pip install pandas numpy matplotlib seaborn scipy factor_analyzer prince
```

## 📥 Importação dos Pacotes

- `pandas` / `numpy` — manipulação de dados
- `matplotlib` / `seaborn` — visualizações
- `factor_analyzer` — PCA e teste de Bartlett
- `scipy.stats.chi2_contingency` — testes qui-quadrado
- `prince` — Análise de Correspondência Múltipla (ACM)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity
from scipy.stats import chi2_contingency
import prince
import warnings
warnings.filterwarnings("ignore")

## 📂 Carregamento do Banco de Dados

- Arquivo: `preco_casas_completo.xlsx`
- Contém variáveis **qualitativas** (binárias e ordinais) e **quantitativas** sobre características de imóveis

In [ ]:
casas = pd.read_excel('preco_casas_completo.xlsx')
casas.head()

---
# 🗺️ Parte 1 — Análise de Correspondência Múltipla (ACM)

## 🔀 Separação das Variáveis Qualitativas

- A ACM opera exclusivamente sobre **variáveis categóricas**
- São selecionadas 7 variáveis: 6 binárias (`yes/no`) e 1 ordinal (`room_size_class`)

In [ ]:
casas_quali = casas[['large_living_room',
                     'parking_space',
                     'front_garden',
                     'swimming_pool',
                     'wall_fence',
                     'water_front',
                     'room_size_class']]

casas_quali.shape

## 📊 Tabelas de Frequências

- Inspeção da distribuição de cada variável qualitativa
- Verifica se há categorias com frequência muito baixa (pode distorcer a ACM)

In [ ]:
for col in casas_quali.columns:
    print(f"\n--- {col} ---")
    print(casas_quali[col].value_counts())

## 🧪 Testes Qui-Quadrado — Associação entre Variáveis

- A ACM exige que haja **associação estatística** entre as variáveis
- Usamos `large_living_room` como referência e testamos todas as demais
- Critério: **p-valor < 0,05** indica associação significativa

In [ ]:
variaveis = ['parking_space', 'front_garden', 'swimming_pool',
             'wall_fence', 'water_front', 'room_size_class']

for var in variaveis:
    tabela = pd.crosstab(casas_quali['large_living_room'], casas_quali[var])
    _, p, _, _ = chi2_contingency(tabela)
    print(f"large_living_room vs {var}: p-valor = {round(p, 4)}")

### Interpretacao

> **p-valor < 0,05** → rejeita-se H₀ de independência → há associação entre as variáveis.  
> Caso alguma variável não apresente associação com a referência, deve-se testá-la em relação a outra variável do conjunto.  
> A confirmação de associação justifica o uso da ACM para capturar a estrutura latente das variáveis qualitativas.

## ⚙️ Elaboração da ACM

- Antes de ajustar o modelo, os `_` nos nomes das variáveis são substituídos por `-`
- Isso é necessário porque a extração de coordenadas do `prince` usa `_` para separar nome da variável da categoria
- `n_components=2` → projeção em 2 dimensões para o mapa perceptual

In [ ]:
# Substituir underline para compatibilidade com prince
casas_quali.columns = casas_quali.columns.str.replace("_", "-")

# Ajustar a ACM
mca = prince.MCA(n_components=2).fit(casas_quali)
print("ACM ajustada com sucesso.")

## 📈 Autovalores da ACM

- Os **autovalores** medem a inércia (variância) capturada por cada dimensão
- Dimensões com maior percentual de inércia são mais informativas

In [ ]:
tabela_autovalores = mca.eigenvalues_summary
print(tabela_autovalores)

### Interpretacao

> Cada linha representa uma dimensão da ACM.  
> A coluna `% of variance` indica quanto da inércia total aquela dimensão explica.  
> Na ACM, ao contrário da PCA, não existe um critério rígido como a raiz latente — a escolha do número de dimensões é guiada pela inércia explicada e pela interpretabilidade do mapa perceptual.

## 📍 Coordenadas-Padrão das Categorias

- As **coordenadas-padrão** posicionam cada categoria das variáveis no espaço dimensional
- Categorias próximas no mapa perceptual tendem a ocorrer juntas nas observações

In [ ]:
coord_padrao = mca.column_coordinates(casas_quali) / np.sqrt(mca.eigenvalues_)
print(coord_padrao)

## 👁️ Coordenadas das Observações

- As **coordenadas das observações** projetam cada linha do dataset no espaço da ACM
- Essas coordenadas serão reutilizadas como variáveis quantitativas na PCA

In [ ]:
coord_obs = mca.row_coordinates(casas_quali)
coord_obs.rename(columns={0: 'dim1_acm', 1: 'dim2_acm'}, inplace=True)
coord_obs.head()

## 🖼️ Mapa Perceptual da ACM

- Visualização das categorias no espaço bidimensional
- Cada cor representa uma variável; os pontos são as categorias
- **Categorias próximas** indicam associação — casas que possuem uma tendem a possuir a outra

In [ ]:
chart = coord_padrao.reset_index()

var_chart = pd.Series(chart['index'].str.split('_', expand=True).iloc[:, 0])

nome_categ = []
for col in casas_quali:
    nome_categ.append(casas_quali[col].sort_values(ascending=True).unique())
    categorias = pd.DataFrame(nome_categ).stack().reset_index()

chart_df_mca = pd.DataFrame({'categoria': chart['index'],
                             'obs_x': chart[0],
                             'obs_y': chart[1],
                             'variavel': var_chart,
                             'categoria_id': categorias[0]})

def label_point(x, y, val, ax):
    a = pd.concat({'x': x, 'y': y, 'val': val}, axis=1)
    for i, point in a.iterrows():
        ax.text(point['x'] + 0.03, point['y'] - 0.02, point['val'], fontsize=5)

plt.figure(dpi=600)

label_point(x=chart_df_mca['obs_x'],
            y=chart_df_mca['obs_y'],
            val=chart_df_mca['categoria_id'],
            ax=plt.gca())

sns.scatterplot(data=chart_df_mca, x='obs_x', y='obs_y', hue='variavel', s=15)
sns.despine(top=True, right=True, left=False, bottom=False)
plt.axhline(y=0, color='lightgrey', ls='--', linewidth=0.8)
plt.axvline(x=0, color='lightgrey', ls='--', linewidth=0.8)
plt.tick_params(size=2, labelsize=6)
plt.legend(bbox_to_anchor=(1.1, -0.15), fancybox=True, shadow=False, ncols=7, fontsize='5')
plt.title("Mapa Perceptual - ACM", fontsize=12)
plt.xlabel(f"Dim. 1: {tabela_autovalores.iloc[0, 1]} da inércia", fontsize=8)
plt.ylabel(f"Dim. 2: {tabela_autovalores.iloc[1, 1]} da inércia", fontsize=8)
plt.show()

### Interpretacao

> O mapa perceptual revela **padrões de associação** entre as categorias das variáveis qualitativas.  
> Categorias posicionadas próximas umas das outras tendem a ocorrer conjuntamente nas mesmas observações.  
> O eixo horizontal (Dim. 1) captura a maior parte da inércia; o eixo vertical (Dim. 2) captura o restante.  
> As coordenadas das observações (`dim1_acm` e `dim2_acm`) serão incorporadas à PCA como variáveis quantitativas.

---
# 📐 Parte 2 — Análise de Componentes Principais (PCA)

## 🔗 Banco de Dados Híbrido — Variáveis Métricas + Coordenadas da ACM

- Separamos as variáveis originalmente quantitativas
- **Concatenamos as coordenadas** extraídas da ACM (`dim1_acm`, `dim2_acm`)
- Resultado: dataset unificado com informação de ambos os tipos de variáveis

In [ ]:
casas_quanti = casas[['land_size_sqm',
                      'house_size_sqm',
                      'no_of_rooms',
                      'no_of_bathrooms',
                      'distance_to_school',
                      'house_age',
                      'distance_to_supermarket_km',
                      'crime_rate_index']]

# Adicionando as coordenadas da ACM
casas_quanti = pd.concat([casas_quanti, coord_obs], axis=1)

print(f"Dimensões: {casas_quanti.shape}")
casas_quanti.head()

## 🧮 Teste de Esfericidade de Bartlett

- Verifica se as variáveis são **suficientemente correlacionadas** para justificar a PCA
- H₀: a matriz de correlações é uma matriz identidade (sem correlação)
- Critério: **p-valor < 0,05** → rejeita H₀ → PCA é adequada

In [ ]:
bartlett, p_value = calculate_bartlett_sphericity(casas_quanti)

print(f'Qui² Bartlett: {round(bartlett, 2)}')
print(f'p-valor: {round(p_value, 4)}')

### Interpretacao

> **p-valor < 0,05** → rejeitamos H₀ → as variáveis possuem correlações significativas entre si.  
> Isso confirma que a PCA é **estatisticamente adequada** para este conjunto de dados.  
> Se o p-valor fosse ≥ 0,05, a PCA não seria recomendada, pois as variáveis seriam praticamente independentes.

## ⚙️ PCA Inicial — Todos os Fatores Possíveis

- Primeiro ajuste com o máximo de fatores (`n_factors=10`) para inspecionar todos os autovalores
- Permite aplicar o **critério da raiz latente** antes de redefinir o modelo final

In [ ]:
fa = FactorAnalyzer(n_factors=10, method='principal', rotation=None).fit(casas_quanti)
print("PCA inicial ajustada.")

## 🔢 Autovalores da PCA

- Os autovalores indicam a variância explicada por cada componente
- A soma dos autovalores é igual ao número de variáveis
- Critério da raiz latente: **autovalor > 1** → componente retido

In [ ]:
autovalores = fa.get_eigenvalues()[0]
print("Autovalores:")
for i, av in enumerate(autovalores):
    print(f"  Componente {i+1}: {round(av, 4)}")

print(f"\nSoma total: {round(autovalores.sum(), 2)}")

## ✂️ Critério da Raiz Latente

- Retemos apenas os fatores com **autovalor > 1**
- Esses fatores explicam mais variância do que uma única variável original
- Fatores com autovalor ≤ 1 são descartados por não agregar informação relevante

In [ ]:
sel_fator = sum(autovalores > 1)
print(f'Quantidade de fatores selecionados (autovalor > 1): {sel_fator}')

### Interpretacao

> O **critério da raiz latente** (Kaiser) define que só são úteis os fatores cujo autovalor supera 1.  
> Isso equivale a dizer que o fator deve explicar mais variância do que a contribuição de qualquer variável individual padronizada.  
> O número de fatores selecionados reduz a dimensionalidade preservando a maior parte da informação.

## 🔄 PCA Final — Fatores Selecionados

- Reajuste do modelo usando apenas os `sel_fator` componentes retidos pelo critério da raiz latente

In [ ]:
fa = FactorAnalyzer(n_factors=sel_fator, method='principal', rotation=None).fit(casas_quanti)
print(f"PCA final ajustada com {sel_fator} fatores.")

## 📊 Eigenvalues, Variâncias e Variâncias Acumuladas

- **Autovalor**: magnitude da variância capturada pelo fator
- **Variância (%)**: proporção da variância total explicada por cada fator
- **Variância Acumulada (%)**: variância explicada pelos fatores até aquele ponto

In [ ]:
autovalores_fatores = fa.get_factor_variance()

tabela_eigen = pd.DataFrame(autovalores_fatores)
tabela_eigen.columns = [f"Fator {i+1}" for i in range(tabela_eigen.shape[1])]
tabela_eigen.index = ['Autovalor', 'Variância', 'Variância Acumulada']
tabela_eigen = tabela_eigen.T

print(tabela_eigen)

### Interpretacao

> Idealmente, os fatores retidos devem explicar ao menos **60–70% da variância total** acumulada.  
> Quanto maior a variância acumulada, mais fiel é a representação fatorial em relação ao dataset original.  
> A variância explicada de cada fator orienta sua interpretação: fatores com maior variância têm mais peso analítico.

## 🔗 Cargas Fatoriais

- As **cargas fatoriais** medem a correlação entre cada variável e cada fator extraído
- Valores absolutos próximos de 1 indicam forte associação
- Cargas **> 0,5** (em módulo) são geralmente consideradas relevantes

In [ ]:
cargas_fatoriais = fa.loadings_

tabela_cargas = pd.DataFrame(cargas_fatoriais)
tabela_cargas.columns = [f"Fator {i+1}" for i in range(tabela_cargas.shape[1])]
tabela_cargas.index = casas_quanti.columns

print(tabela_cargas.round(4))

### Interpretacao

> As cargas fatoriais revelam quais variáveis originais **definem** cada fator.  
> Um fator é interpretado pelo conjunto de variáveis com **cargas absolutas > 0,5**.  
> Variáveis com carga próxima de zero em todos os fatores contribuem pouco para a estrutura fatorial.  
> Sinais positivos e negativos indicam a direção da relação com o fator.

## 🧩 Comunalidades

- A **comunalidade** de uma variável é a proporção de sua variância explicada pelos fatores retidos
- Varia de 0 a 1: valores próximos de **1** indicam que a variável é bem representada pelos fatores
- Valores baixos sugerem que a variável não é bem capturada pela estrutura fatorial

In [ ]:
comunalidades = fa.get_communalities()

tabela_comunalidades = pd.DataFrame(comunalidades,
                                    columns=['Comunalidades'],
                                    index=casas_quanti.columns)

print(tabela_comunalidades.round(4))

### Interpretacao

> Comunalidades **próximas de 1** → a variável é amplamente explicada pelos fatores.  
> Comunalidades **abaixo de 0,5** → a variável tem variância específica significativa não capturada pelos fatores.  
> Na PCA sem rotação, as comunalidades tendem a ser altas quando o número de fatores retidos é adequado.

## 🏗️ Extração dos Fatores para as Observações

- Calculamos os **escores fatoriais** de cada observação — sua posição no espaço fatorial
- Os fatores são adicionados ao dataset original para análises futuras (ex.: ranking, segmentação)

In [ ]:
fatores = pd.DataFrame(fa.transform(casas_quanti))
fatores.columns = [f"Fator {i+1}" for i in range(fatores.shape[1])]

casas = pd.concat([casas.reset_index(drop=True), fatores], axis=1)

print(f"Dataset final: {casas.shape}")
casas[fatores.columns].head()

## 🏋️ Scores Fatoriais

- Os **scores** (pesos) indicam como cada variável contribui para o cálculo do escore de cada observação
- São os coeficientes da combinação linear que gera os fatores
- Diferem das cargas fatoriais: cargas = correlação; scores = coeficientes de ponderação

In [ ]:
scores = fa.weights_

tabela_scores = pd.DataFrame(scores)
tabela_scores.columns = [f"Fator {i+1}" for i in range(tabela_scores.shape[1])]
tabela_scores.index = casas_quanti.columns

print(tabela_scores.round(4))

## 📉 Visualização dos Scores Fatoriais

- Gráfico de barras comparando os pesos de cada variável em cada fator
- Facilita a **interpretação** de quais variáveis mais influenciam cada fator
- Variáveis com scores elevados (positivos ou negativos) são as mais determinantes

In [ ]:
tabela_scores_graph = tabela_scores.reset_index()
tabela_scores_graph = tabela_scores_graph.melt(id_vars='index')

plt.figure(figsize=(12, 8), dpi=600)
sns.barplot(data=tabela_scores_graph, x='variable', y='value', hue='index', palette='viridis')
plt.legend(title='Variáveis', bbox_to_anchor=(1, 1), fontsize='6')
plt.title('Scores Fatoriais', fontsize=12)
plt.xlabel(None)
plt.ylabel(None)
plt.show()

### Interpretacao

> O gráfico de scores fatoriais permite **nomear** cada fator com base nas variáveis mais influentes.  
> Barras de maior magnitude (positiva ou negativa) indicam as variáveis que mais contribuem para diferenciar as observações naquele fator.  
> Por exemplo, se `house_size_sqm` e `no_of_rooms` dominam o Fator 1, ele pode ser interpretado como **"porte do imóvel"**.  
> Os escores fatoriais de cada observação podem ser usados em análises subsequentes, como ranqueamento ou segmentação.

---
# ✅ Conclusão

Esta análise híbrida combinou:

| Etapa | Técnica | Objetivo |
|-------|---------|----------|
| 1 | **ACM** | Estruturar variáveis qualitativas em dimensões latentes |
| 2 | **PCA** | Reduzir dimensionalidade unindo variáveis métricas + coordenadas ACM |

Os fatores extraídos resumem as características dos imóveis em poucas dimensões interpretáveis, eliminando redundâncias e preservando a estrutura da informação.